In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path = [p for p in sys.path if 'inftools' not in p]
sys.path.insert(0, os.path.abspath('..'))
import matplotlib.pyplot as plt
import numpy as np
import deeptime as dpt
%matplotlib qt   
# doesn't work on my laptop

from tistools import read_inputfile, get_LMR_interfaces, read_pathensemble, get_weights
from tistools import set_tau_distrib, set_tau_first_hit_M_distrib, cross_dist_distr, pathlength_distr
from tistools import ACCFLAGS, REJFLAGS

from tistools import get_lmr_masks, get_generation_mask, get_flag_mask, select_with_masks
from tistools import unwrap_by_weight, running_avg_local_probs, get_local_probs, get_global_probs_from_dict, get_global_probs_from_local
from tistools import make_plot_trajs

from pprint import pprint    # to print the vars of the pathensemble object

# TODO: change to istar_analysis when finished
# from istar_test import *
from tistools import global_pcross_msm_star, construct_M_istar, get_transition_probs_weights, get_transition_probs_interm, get_simple_probs, get_summed_probs
from tistools import get_weights_staple, compute_weight_matrices, compute_weight_matrix, plot_rv_star, plot_rv_repptis, plot_rv_comp
from tistools import display_data, memory_analysis, ploc_memory, plot_memory_analysis, pcca_analysis, cprobs_repptis_istar, ploc_repptis_from_staples, generate_state_labels, visualize_turn_based_analysis, analyze_memory_vs_free_energy_effects, analyze_momentum_vs_free_energy


%matplotlib qt

In [2]:
import logging

logger = logging.getLogger(__name__)

In [3]:
%autoreload 2   
# something with pip install -e .

# Reading

In [6]:
# zero_minus_one = True if lambda_-1 interface is set
# zero_minus_one = False if lambda_-1 interface is not set

# data the maze
# ---------------
indir = "/mnt/0bf0c339-34bb-4500-a5fb-f3c2a863de29/DATA/APPTIS/simdata_kcl/infrepptis/"
indir = "/run/user/1001/gvfs/smb-share:server=files.ugent.be,share=eliawils,user=eliawils/shares/tw06_biommeda_pastime1/11.2024_StapleTIS_Elias/simulations/Z_pot2D/sim_istarz_g5/"
indir = "/run/user/1001/gvfs/smb-share:server=files.ugent.be,share=eliawils,user=eliawils/shares/tw06_biommeda_pastime1/11.2024_StapleTIS_Elias/simulations/cos_well/sim_istarwell2rv/"

zero_minus_one = False
inputfile = indir + "logging.log"
# inputfile = indir + "fakeretis.rst"
# inputfile = indir + "repptis.rst" # fake repptis.rst file with interfaces

import os
import glob
os.chdir(indir)
print(os.getcwd())

folders = glob.glob(indir + "/0[0-9][0-9]")
folders = sorted(folders)
print(folders)

/run/user/1001/gvfs/smb-share:server=files.ugent.be,share=eliawils,user=eliawils/shares/tw06_biommeda_pastime1/11.2024_StapleTIS_Elias/simulations/cos_well/sim_istarwell2rv
['/run/user/1001/gvfs/smb-share:server=files.ugent.be,share=eliawils,user=eliawils/shares/tw06_biommeda_pastime1/11.2024_StapleTIS_Elias/simulations/cos_well/sim_istarwell2rv/000', '/run/user/1001/gvfs/smb-share:server=files.ugent.be,share=eliawils,user=eliawils/shares/tw06_biommeda_pastime1/11.2024_StapleTIS_Elias/simulations/cos_well/sim_istarwell2rv/001', '/run/user/1001/gvfs/smb-share:server=files.ugent.be,share=eliawils,user=eliawils/shares/tw06_biommeda_pastime1/11.2024_StapleTIS_Elias/simulations/cos_well/sim_istarwell2rv/002', '/run/user/1001/gvfs/smb-share:server=files.ugent.be,share=eliawils,user=eliawils/shares/tw06_biommeda_pastime1/11.2024_StapleTIS_Elias/simulations/cos_well/sim_istarwell2rv/003', '/run/user/1001/gvfs/smb-share:server=files.ugent.be,share=eliawils,user=eliawils/shares/tw06_biommeda_pas

In [9]:
# !!! last lines !!!  allow to speed up this notebook
# pe.set_orders(load=False...)  -> 1st time you run the code, this will store npy files
# pe.set_orders(load=True...)  -> next time you run the code, you can read npy files

# Reading all input
#===================
interfaces, zero_left, timestep = read_inputfile(inputfile)
LMR_interfaces, LMR_strings = get_LMR_interfaces(interfaces, zero_left)
pathensembles = []
for i,fol in enumerate(folders):
    print("#"*80)
    print(fol)
    pe = read_pathensemble(fol+"/pathensemble.txt")
    pe.set_name(fol)
    pe.set_interfaces([LMR_interfaces[i], LMR_strings[i]])
    if i==0:
        pe.set_zero_minus_one(zero_minus_one)   # TODO this is never used
        pe.set_in_zero_minus(True)
    if i==1:
        pe.set_in_zero_plus(True)
    w, _ = get_weights(pe.flags, ACCFLAGS, REJFLAGS, verbose = False)
    pe.set_weights(w)
    print("pathensemble info: ")
    pprint(vars(pe))
    pathensembles.append(pe)
    # read order parameters order.txt/order.npy into path ensemble object
    #pe.set_orders(load=False, acc_only=True, save=False) # if saving doesn't work
    #### CHANGE HERE ####
    # pe.set_orders(load=False, acc_only=True, save=True) # for the 1st time
    pe.set_orders(load=True, acc_only=True) # for the next times, save=True/False is not important

################################################################################
/run/user/1001/gvfs/smb-share:server=files.ugent.be,share=eliawils,user=eliawils/shares/tw06_biommeda_pastime1/11.2024_StapleTIS_Elias/simulations/cos_well/sim_istarwell2rv/000
pathensemble info: 
{'cyclenumbers': array([     0,      1,      2, ...,  99998,  99999, 100000]),
 'dirs': array([-1., -1., -1., ..., -1., -1.,  1.]),
 'flags': array(['ACC', 'FTL', 'FTL', ..., 'FTL', 'ACC', 'ACC'], dtype='<U3'),
 'generation': array(['ld', 'sh', 'sh', ..., 'sh', 'sh', 'sh'], dtype='<U2'),
 'has_zero_minus_one': False,
 'in_zero_minus': True,
 'in_zero_plus': False,
 'interfaces': [[-0.35, -0.35, -0.35], ['l_[0]', 'l_[0]', 'l_[0]']],
 'istar_idx': array([[0, 0],
       [0, 0],
       [0, 0],
       ...,
       [0, 0],
       [0, 0],
       [0, 0]]),
 'lambmaxs': array([-0.34864259, -0.34870599, -0.34958609, ..., -0.34956581,
       -0.34713912, -0.34554693]),
 'lambmins': array([-0.49490423, -0.5235762 , -0.5013851

In [11]:
C, X_tr, X_norm, X_tr_norm, wpath, W_tr, W_norm, W_tr_norm, W = display_data(pathensembles, interfaces, len(interfaces), correct_ha=False)
p, q = get_transition_probs_weights(wpath)

----------
ENSEMBLE [0-] | ID 0
----------
None ha_factors for 0th ensemble
None
weights:
accepted      59613
rejected      40388
omitted       0
total trajs   100001
total weights 100001
None ha_factors for 0th ensemble
None
Sum weights ensemble 0: 0.0000
None ha_factors for 0th ensemble
None
Sum weights ensemble 0: 0.0000

1a. Raw data: unweighted C matrices
C[0] = 
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]

1b. Raw data: unweighted path counts with new MD steps
C_md[0] = 
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0

In [12]:
# q_k, q_tot = memory_analysis(wpath, False)
# plot_memory_analysis(pathensembles, q_tot, p, interfaces, q_errors=indir + "block_error_analysis_qstaple_1.txt")
# plot_memory_analysis(pathensembles, q_tot, p, interfaces)
# plot_free_energy_landscape(interfaces, q_tot[0], q_tot[1])


In [10]:
ploc_memory(pathensembles, interfaces, trr=False)

Weights of the different paths:
wRMR = 99998
wRML = 0
wLMR = 0
wLML = 0
Local crossing probabilities:
pRMR = 1.0
pRML = 0.0
pLMR = nan
pLML = nan
Local crossing probabilities:
p2R = 1.0
p2L = 0.0
None ha_factors for 0th ensemble
None
weights:
accepted      77412
rejected      22589
omitted       0
total trajs   100001
total weights 100001
Sum weights ensemble 0:  0.0
None ha_factors for 1th ensemble
None
weights:
accepted      58832
rejected      41169
omitted       0
total trajs   100001
total weights 100001
Sum weights ensemble 1:  100000.0

Intermediate transition probabilities (q matrix):
[[1.     0.8214]
 [1.     0.    ]]

Final transition probabilities (p matrix):
[[0.1786 0.8214]
 [1.     0.    ]]

Local crossing probabilities computed successfully
Weights of the different paths:
wRMR = 0
wRML = 45396
wLMR = 44851
wLML = 9753
Local crossing probabilities:
pRMR = 0.0
pRML = 1.0
pLMR = 0.82138671159622
pLML = 0.17861328840377994
Local crossing probabilities:
p2R = 0.44851
p2L = 0.

{'mlst': [1.0,
  0.82138671159622,
  0.2242505149,
  0.1733023397399375,
  0.12488754677615685,
  0.09713104361452546],
 'apptis': [1.0,
  0.82138671159622,
  0.8063825027667776,
  0.790976068476316,
  0.6435206723427925,
  0.632629994980046],
 'apptis_ha': [1.0],
 'apptis_ha, norm': [1.0],
 'repptis': [1.0,
  0.82138671159622,
  0.8061946432922809,
  0.7875235546443652,
  0.5576545654549856,
  0.5462433103717984]}

In [115]:
M = construct_M_istar(p, 2*len(interfaces), len(interfaces))
print(np.sum(M, axis=1))

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [91]:
# get stationary distrib
#------------------------
def get_summation_distr(M):
    # This is actually an eigenvector of M, with just ones in it.
    a = M-np.identity(len(M))
    # this would just be a constanct vector
    vals, vecs = np.linalg.eig(a)
    print(vals)
    order = np.argsort(abs(vals))
    vals = vals[order]
    print(vals)
    vec = vecs[:,order[0]]
    print(vec)

def get_stationary_distr(M):
    """Calculate stationary distr of M
    First substract identity matrix, such that
    the stationary vector is the LEFT eigenvecor of M
    corresponding with eigenvalue 0"""
    # for i in range(len(M)):
    #     if np.sum(M[:,i])==0:
    #         M = np.delete(np.delete(M, i, axis=0), i, axis=1)
    a = M-np.identity(len(M))

    vals, vecs = np.linalg.eig(a.T)
    #print(vals)
    order = np.argsort(abs(vals))
    vals = vals[order]
    #print(vals)
    vec = vecs[:,order[0]]
    #print(vec)

    # make vector positive
    if max(vec) < 0:
        vec *= -1
    # remove imaginary part of vector
    if np.sum(abs(vec-vec.real)) < 1e-8:
        vec = vec.real
    # all elements should be larger than zero
    assert vec.all()>=0
    #assert abs(np.sum(vec**2)-1) < 1e-8 # normalization is automatic
    # renormalize to sum=1
    vec /= np.sum(vec)
    return vec

summ_d = get_summation_distr(M)
stat_d = get_stationary_distr(M)
print("summation distr", summ_d)
print("stationary distr", stat_d)

[-1.00000000e+00+0.j          2.56025197e-16+0.j
 -1.38394286e+00+0.81343856j -1.38394286e+00-0.81343856j
 -1.23594861e+00+0.j         -1.11602664e+00+0.j
 -9.06581667e-01+0.j         -1.02257212e+00+0.j
 -1.00000000e+00+0.j         -9.50985243e-01+0.j        ]
[ 2.56025197e-16+0.j         -9.06581667e-01+0.j
 -9.50985243e-01+0.j         -1.00000000e+00+0.j
 -1.00000000e+00+0.j         -1.02257212e+00+0.j
 -1.11602664e+00+0.j         -1.23594861e+00+0.j
 -1.38394286e+00+0.81343856j -1.38394286e+00-0.81343856j]
[-0.31622777+0.j -0.31622777+0.j -0.31622777+0.j -0.31622777+0.j
 -0.31622777+0.j -0.31622777+0.j -0.31622777+0.j -0.31622777+0.j
 -0.31622777+0.j -0.31622777+0.j]
summation distr None
stationary distr [ 0.31082588  0.11185053  0.31082588  0.00261682  0.00152588 -0.
  0.00448435  0.05391998  0.00497534  0.19897535]


In [92]:
# Create state labels for better readability
N = len(interfaces)
labels = generate_state_labels(N)

# Get different color/alpha combinations for each run
color_pairs = [('darkgreen', 'darkblue'), 
                ('purple', 'maroon'), ('orange', 'navy')]  # Add more pairs if needed
alpha_values = [0.5, 0.6, 0.7, 0.8]  # Different alpha values for different runs

# Get the current axis if it exists, otherwise create new figure
if plt.get_fignums():
    # Get the current figure
    fig = plt.gcf()
    ax1, ax2 = fig.axes
    
    # Use the next available color pair and alpha value
    used_runs = len([ax for ax in fig.axes if ax.patches])
    bar_color1, bar_color2 = color_pairs[used_runs % len(color_pairs)]
    alpha_val = alpha_values[used_runs % len(alpha_values)]
else:
    # Create a figure with two subplots for the first run
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), gridspec_kw={'height_ratios': [1, 1]})
    bar_color1, bar_color2 = color_pairs[0]
    alpha_val = alpha_values[0]

# Plot the stationary distribution as a bar chart with transparency
ax1.bar(range(len(stat_d)), stat_d, color=bar_color1, alpha=alpha_val, 
        label=f"{indir.split('/')[-2]}")
ax1.set_xticks(range(len(stat_d)))
ax1.set_xticklabels(labels, rotation=45, ha='right')
ax1.set_xlabel("State")
ax1.set_ylabel("Stationary Probability")
ax1.set_title("Stationary Distribution")
ax1.grid(axis='y', linestyle='--', alpha=0.7)
ax1.legend()

# Calculate the free energy: -ln(prob)
# Handle zero probabilities by setting them to NaN to avoid -inf
free_energy = -np.log(np.where(stat_d > 0, stat_d, np.nan))

# Plot the free energy with transparency
ax2.bar(range(len(stat_d)), free_energy, color=bar_color2, alpha=alpha_val, 
        label=f"{indir.split('/')[-2]}")
ax2.set_xticks(range(len(stat_d)))
ax2.set_xticklabels(labels, rotation=45, ha='right')
ax2.set_xlabel("State")
ax2.set_ylabel("Free Energy (-ln(P))")
ax2.set_title("Free Energy Landscape")
ax2.grid(axis='y', linestyle='--', alpha=0.7)
ax2.legend()

# Improve layout
plt.tight_layout()
plt.show()


In [93]:
print(stat_d[-1]/(stat_d[-2]+stat_d[-1]))
print(stat_d[-3]/np.sum(stat_d[-3:]))
print((np.sum(stat_d[2:len(interfaces)-1])/np.sum(stat_d[2:len(interfaces)+1]))*stat_d[-3]/np.sum(stat_d[-3:]))
print((np.sum(stat_d[2:len(interfaces)-2])/np.sum(stat_d[2:len(interfaces)+1]))*stat_d[-4]/np.sum(stat_d[-4:]))
print(stat_d[-3]/(stat_d[-3]+stat_d[-2]*(np.sum(stat_d[2:len(interfaces)-1])/np.sum(stat_d[2:len(interfaces)]))+stat_d[-1]*(np.sum(stat_d[2:len(interfaces)-1])/np.sum(stat_d[2:len(interfaces)+1]))))
print(stat_d[-3]/np.sum(stat_d[-3:]))
print(stat_d[4]/(stat_d[3]+stat_d[4]+stat_d[1]))

for i in range(3,len(pathensembles)+1):
    print(f"relative stat_d, turn fw {i-2}: {stat_d[i]/(stat_d[1]+np.sum(stat_d[2:i+1]))}")
for i in range(len(pathensembles)+1,2*len(pathensembles)-1):
    print(f"relative stat_d, turn bw {i-len(pathensembles)}: {stat_d[i]/np.sum(stat_d[i:])}")

0.9756051815473528
0.2090969856416371
0.20808400427492593
0.016867852878574296
0.20990123720529524
0.2090969856416371
0.013154921952176242
relative stat_d, turn fw 1: 0.0061529792191542705
relative stat_d, turn fw 2: 0.003575008377561979
relative stat_d, turn fw 3: -0.0
relative stat_d, turn bw 1: 0.01709266843392905
relative stat_d, turn bw 2: 0.2090969856416371
relative stat_d, turn bw 3: 0.024394818452647224


# Analysis

In [116]:
# Analyze the [i*] simulation.
# Analysis output is saved to the data dictionary.
data = {}

data["running"] = {}

# analysis using all data: ["full"]
# data["full"] = get_transition_probs(pathensembles, interfaces)
# pprint(data)
# print("\n\n")
# data["full"] = get_transition_probzz2(pathensembles, interfaces)
w = compute_weight_matrices(pathensembles, interfaces, tr=False)
for i in range(len(pathensembles)):
    print(f"sum weights pe {i}: ",np.sum(w[i]))

data["full"], _ = get_transition_probs_weights(w)
# data["full"] = get_transition_probs(w)
# data["full1"] = get_simple_probs(w)
# data["full"] = get_summed_probs(pathensembles, interfaces)
pprint(data)

None ha_factors for 0th ensemble
None
weights:
accepted      60538
rejected      39463
omitted       0
total trajs   100001
total weights 100001
Sum weights ensemble 0:  0.0
None ha_factors for 1th ensemble
None
weights:
accepted      65157
rejected      34844
omitted       0
total trajs   100001
total weights 100001
Sum weights ensemble 1:  99997.0
None ha_factors for 2th ensemble
None
weights:
accepted      48813
rejected      51188
omitted       0
total trajs   100001
total weights 106609
Sum weights ensemble 2:  135695.0
None ha_factors for 3th ensemble
None
weights:
accepted      53706
rejected      46295
omitted       0
total trajs   100001
total weights 135553
Sum weights ensemble 3:  135552.0
None ha_factors for 4th ensemble
None
weights:
accepted      52405
rejected      47596
omitted       0
total trajs   100001
total weights 140993
Sum weights ensemble 4:  140992.0
None ha_factors for 5th ensemble
None
weights:
accepted      49996
rejected      50005
omitted       0
total tr

In [117]:
# Make a figure of the global crossing probabilities
# fig, ax = plt.subplots()
# ax.set_yscale("log")
# ax.plot(Pcrossfull, "o", c = "r")

# cosdip meta
# ax.errorbar([i for i in range(7)], Pcrossfull, yerr=[0, 0.004830, Pcrossfull[2]*0.05068988646, Pcrossfull[3]*0.05189862680, Pcrossfull[4]*0.05071184896, Pcrossfull[5]*0.05083284286, Pcrossfull[6]*0.05067963543], fmt="-o", c = "b", ecolor="r", capsize=6)

# cosbump meta
# ax.errorbar([i for i in range(7)], Pcrossfull, yerr=[0, 0.002535, Pcrossfull[2]*0.04393065503, Pcrossfull[3]*0.04910273500, Pcrossfull[4]*0.05239942040, Pcrossfull[5]*0.05789033634, Pcrossfull[6]*0.0614468], fmt="-o", c = "b", ecolor="r", capsize=6)

# 2 cosdips
# ax.errorbar([i for i in range(5)], Pcrossfull, yerr=[0, 0.007239, Pcrossfull[2]*0.0414296, Pcrossfull[3]*0.0445266, Pcrossfull[4]*0.0483538], fmt="-o", c = "b", ecolor="r", capsize=6)

# 3 cosbumps
# ax.errorbar([i for i in range(7)], Pcrossfull, yerr=[0, 0.002295, Pcrossfull[2]*0.0328798, Pcrossfull[3]*0.031594, Pcrossfull[4]*0.031474, Pcrossfull[5]*0.03080392, Pcrossfull[6]*0.0308589], fmt="-o", c = "b", ecolor="r", capsize=6)

# 2 cosbumps
#ax.errorbar([i for i in range(5)], Pcrossfull, yerr=[0, 0.002768, Pcrossfull[2]*0.04440278, Pcrossfull[3]*0.043053, Pcrossfull[4]*0.0463156], fmt="-o", c = "b", ecolor="r", capsize=6)

# flat dt=0.00002 30k cycles
# ax.errorbar([i for i in range(5)], Pcrossfull, yerr=[0, 0.003294, Pcrossfull[2]*0.07640968, Pcrossfull[3]*0.07789262, Pcrossfull[4]*0.0812692], fmt="-o", c = "b", ecolor="r", capsize=6)

# flat 100k cycles
#ax.errorbar([i for i in range(5)], Pcrossfull, yerr=[0, 0.002741, Pcrossfull[2]*0.034092, Pcrossfull[3]*0.033621, Pcrossfull[4]*0.0398], fmt="-o", c = "b", ecolor="r", capsize=6)

# ax.set_xlabel("intf")
# ax.set_ylabel(r"$P_A(\lambda_i|\lambda_A)$")
# ax.set_xticks(np.arange(len(interfaces)))
# fig.tight_layout()
# fig.show()
# fig.savefig("Global_probs.pdf")

# print("This should be the same as the repptis_report.pdf value:", Pcrossfull[-1])
# print("which is the case!")
# print("Here, the load immediately disappeared. For a simulation where this is")
# print("not the case, the above code should be adapted a little bit.")

# Now work with MSM

In [118]:
from tistools import construct_M
from tistools import global_pcross_msm
from tistools import mfpt_to_first_last_state

from tistools import create_labels_states

In [119]:
def print_vector(g, states=None):
    if states is None:
        for i in range(len(g)):
            print("state", i, g[i])
    else:
        for i in range(len(g)):
            print("state", states[i], g[i][0])

In [120]:
print(interfaces)
N = len(interfaces)
assert N >= 3
NS = 2*N
print("N", N)
print("NS", NS)

#labels2 = ["0+- LML","0+- LMR","0+- RML","1+- LML","1+- LMR",
#           "1+- RML", "1+- RMR", "2+- LML", "2+- LMR",
#           "2+- RML", "2+- RMR", "3+- LML", "3+- LMR",]
labels1, labels2 = create_labels_states(N)
print(labels1, labels2)

[0.05, 0.13, 0.21, 0.29, 0.37, 0.45, 0.53, 0.61, 0.69, 0.77, 0.85]
N 11
NS 22
['0-     ', 'B      '] ['0+- LML', '0+- LMR', '0+- RML', '1+- LML', '1+- LMR', '1+- RML', '1+- RMR', '2+- LML', '2+- LMR', '2+- RML', '2+- RMR', '3+- LML', '3+- LMR', '3+- RML', '3+- RMR', '4+- LML', '4+- LMR', '4+- RML', '4+- RMR', '5+- LML', '5+- LMR', '5+- RML', '5+- RMR', '6+- LML', '6+- LMR', '6+- RML', '6+- RMR', '7+- LML', '7+- LMR', '7+- RML', '7+- RMR', '8+- LML', '8+- LMR', '8+- RML', '8+- RMR', '9+- LML', '9+- LMR']


In [121]:
p_ini = data["full"]
print("p matrix: ", p_ini)
p_ini[-2][-1] = 1
print("sum rows of p:")
for i in range(p_ini.shape[0]):
    print(np.sum(p_ini[i][:i]), np.sum(p_ini[i][i:]))
M_valid = construct_M_istar(p_ini, NS, N)
# M1 = construct_M_istar(data["full1"], NS, N)

# for r in range(M.shape[0]):
#     if np.sum(M[r]) != 0:
#         M[r] /= np.sum(M[r])
#Local crossing probabilities:
#pRMR = 0.34205627942625644.  #ppps
#pRML = 0.6579437205737436.   #ppms
#pLMR = 0.25316455696202533.  #pmps
#pLML = 0.7468354430379747.   #pmms

p matrix:  [[9.52165603e-01 2.28419390e-02 1.21677670e-02 7.28951853e-03
  3.40850470e-03 1.62655468e-03 3.59502946e-04 4.76468245e-05
  4.08016158e-05 1.65305859e-05 3.56310026e-05]
 [1.00000000e+00 0.00000000e+00 4.94481364e-01 2.89056715e-01
  1.33518568e-01 6.32549445e-02 1.46292780e-02 1.72967563e-03
  1.39579975e-03 6.10094477e-04 1.32356044e-03]
 [8.18501361e-01 1.81498639e-01 0.00000000e+00 5.62109449e-01
  2.59908368e-01 1.33574080e-01 3.19981071e-02 4.03161888e-03
  3.35557196e-03 1.77490166e-03 3.24790416e-03]
 [6.56997724e-01 1.42268817e-01 2.00733459e-01 0.00000000e+00
  6.17196488e-01 2.96898337e-01 6.45616920e-02 8.04130139e-03
  5.26836399e-03 2.46670224e-03 5.56711527e-03]
 [5.39509521e-01 1.13868686e-01 1.66305701e-01 1.80316092e-01
  0.00000000e+00 3.67073433e-01 2.43517056e-01 1.42683784e-01
  1.04056410e-01 4.83136390e-02 9.43556781e-02]
 [4.22497998e-01 9.54292242e-02 1.39014913e-01 1.44054004e-01
  1.99003860e-01 0.00000000e+00 3.25626917e-01 2.48590849e-01
  1.7

In [122]:
print("M")
print("shape", M_valid.shape)
print("sum prob in rows", np.sum(M_valid,axis=1))
print(M_valid)
print(np.max(abs(M-M_valid)))
# print(M1)
# row 8, 10, 12, 14. # counting starts from 0   not okay!!!!

M
shape (22, 22)
sum prob in rows [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
[[0.00000000e+00 0.00000000e+00 1.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [1.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 9.52165603e-01 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  2.28419390e-02 1.2167767

# Look at this Markov model

In [123]:
#import numpy.linalg
vals, vecs = np.linalg.eig(M_valid)
print(vals)
vals, vecs = np.linalg.eig(M_valid.T)
print(vals)

[ 0.00000000e+00+0.j        -4.78656033e-01+0.8532706j
 -4.78656033e-01-0.8532706j  1.00000000e+00+0.j
  7.94493364e-01+0.j        -7.95520678e-01+0.j
  5.89524632e-01+0.j        -5.99640190e-01+0.j
  4.03476012e-01+0.j        -4.05096244e-01+0.j
  3.21383590e-01+0.j         2.62959956e-01+0.j
  2.31209800e-01+0.j         2.18226274e-01+0.j
  1.71229076e-01+0.j        -3.25164855e-01+0.j
 -2.63612778e-01+0.j        -2.31583618e-01+0.j
 -2.19419975e-01+0.j        -1.71230293e-01+0.j
 -3.08812998e-16+0.j        -2.39220068e-02+0.j       ]
[-4.78656033e-01+0.8532706j -4.78656033e-01-0.8532706j
  1.00000000e+00+0.j         7.94493364e-01+0.j
 -7.95520678e-01+0.j         5.89524632e-01+0.j
 -5.99640190e-01+0.j         4.03476012e-01+0.j
 -4.05096244e-01+0.j         3.21383590e-01+0.j
  2.62959956e-01+0.j         2.31209800e-01+0.j
  2.18226274e-01+0.j         1.71229076e-01+0.j
 -3.25164855e-01+0.j        -2.39220068e-02+0.j
 -2.63612778e-01+0.j        -2.31583618e-01+0.j
 -2.19419975e-01+0

In [124]:
print("what if chain propagates")
print("A[0,:]")
# check stationary behavior
A = M_valid
for n in range(10):
    A = np.dot(A,M_valid)
    #print(A)
    print(A[0,:])
    print(np.sum(A[0,:]))  # is 1 indeed

what if chain propagates
A[0,:]
[0.00000000e+00 9.52165603e-01 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 2.28419390e-02 1.21677670e-02 7.28951853e-03 3.40850470e-03
 1.62655468e-03 3.59502946e-04 4.76468245e-05 4.08016158e-05
 1.65305859e-05 3.56310026e-05]
0.9999999999999999
[9.52201234e-01 4.01948022e-02 0.00000000e+00 3.80600835e-03
 2.28385881e-03 8.75257928e-04 4.22571280e-04 1.59499611e-04
 3.32403462e-05 1.90001653e-05 4.52716873e-06 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00]
0.9999999999999999
[4.01948022e-02 0.00000000e+00 9.52201234e-01 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 1.88200020e-03 2.38393089e-03 1.64197292e-03
 9.60790499e

# Pcross with MSM

In [125]:
# global crossing prob
z1, z2, y1, y2 = global_pcross_msm_star(M_valid, True)
print("Z")
print_vector(z1, labels1)
print_vector(z2, labels2)
print("Y")
print_vector(y1, labels1)
print_vector(y2, labels2)
print("global crossing prob", y1[0])


=== Eigenvalue Analysis ===
Mp eigenvalues: [ 0.      0.7955 -0.7955  0.595  -0.595   0.4046  0.323   0.2634  0.2313
  0.2188  0.1712 -0.4046 -0.323  -0.2634 -0.1712 -0.2313 -0.2188  0.
  0.    ]
(I-Mp) eigenvalues: [1.     1.7955 0.2045 1.595  0.405  1.4046 1.323  1.2634 1.2313 1.2188
 1.1712 0.5954 0.677  0.7366 0.8288 0.7687 0.7812 1.     1.    ]

=== Matrix Components ===
D (transitions from intermediate to boundary states):
[[0.     0.    ]
 [0.     0.0013]
 [0.     0.0032]
 [0.     0.0056]
 [0.     0.0944]
 [0.     0.1638]
 [0.     0.2494]
 [0.     0.3959]
 [0.     0.656 ]
 [0.     1.    ]
 [0.     0.    ]
 [0.     0.    ]
 [0.     0.    ]
 [0.     0.    ]
 [0.     0.    ]
 [0.     0.    ]
 [0.     0.    ]
 [0.     0.    ]
 [0.     0.    ]]

E (transitions from boundary to intermediate states):
[[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]

M11 (transitions between boundary states):
[0. 0.]

=== Solution 

In [126]:
# P_loc with MSM
plocMSM = np.ones(len(interfaces))
# p2 = [1.0, 0.056804159591634415, 0.032312997136678026, 0.018071661236763906, 0.009174080897093834, 0.004218791762213937, 0.0019851528367788125, 0.0015222409020198072, 0.001318390278132916, 0.0011178599945945473, 0.0009802371708142466]
# p2 = [1.0, 0.8272598636455519, 0.8118884810440299, 0.7938415404869816, 0.5663700466714486, 0.5545303803303038]
# p2 = [1.0, 0.055139270922254895, 0.03126608951116539, 0.017392262443045026, 0.008894249404190929, 0.0040038338896561795, 0.0010513581251145503, 0.0010513581251145503, 0.0010513581251145503, 0.0010513581131851855, 0.0010513580976654149]
# p3 = [1.0, 0.062089, 0.035751653, 0.02010748, 0.010213876, 0.004782699, 0.001383816, 0.000568035, 0.000547059, 0.000516629, 0.000493516]
# p3 = [1.0, 0.810296, 0.796242226, 0.78048937, 0.636988594, 0.625611341]
# p3 = [1.0, 0.059440, 0.575552, 0.565633, 0.517311, 0.461832, 0.259615, 0.368699, 0.992271, 0.999820, 0.987496]
# p3 = [np.prod(p3[:i+1]) for i in range(len(p3))]

for lint in range(2, len(interfaces)+1):
    wi = compute_weight_matrices(pathensembles[:lint], interfaces[:lint], len(interfaces), tr=False)
    pi, _ = get_transition_probs_weights(wi)
    # pi = get_transition_probs(wi)
    # pi = get_simple_probs(wi)
    Mi = construct_M_istar(pi, max(4, 2*len(interfaces[:lint])), len(interfaces[:lint]))
    z1, z2, y1, y2 = global_pcross_msm_star(Mi)
    plocMSM[lint-1] = y1[0][0]
    print(f"ploc till intf {lint-1}: ", y1[0][0])

# Make a figure of the global crossing probabilities
plt.rcParams['text.usetex'] = True
fig, ax = plt.subplots()
ax.set_yscale("log")
ax.plot(plocMSM, "o", c = "r")
ax.errorbar([i for i in range(len(plocMSM))], plocMSM, fmt="-o", c = "b", ecolor="r", capsize=6, label="StapleTIS")
# ax.errorbar([i for i in range(len(plocMSM))], p2, fmt="-o", c = "orange", ecolor="r", capsize=6., label="REPPTIS")
# ax.errorbar([i for i in range(len(plocMSM))], p3, fmt="-o", c = "r", ecolor="r", capsize=6., label="RETIS")
ax.set_xlabel(r"Interface index")
ax.set_ylabel(r"$P_A(\lambda_i|\lambda_A)$")
ax.set_xticks(np.arange(len(interfaces)))
fig.tight_layout()
fig.legend()
fig.show()


None ha_factors for 0th ensemble
None
weights:
accepted      60538
rejected      39463
omitted       0
total trajs   100001
total weights 100001
Sum weights ensemble 0:  0.0
None ha_factors for 1th ensemble
None
weights:
accepted      65157
rejected      34844
omitted       0
total trajs   100001
total weights 100001
Sum weights ensemble 1:  99997.0

Intermediate transition probabilities (q matrix):
[[1.     0.0478]
 [1.     0.    ]]

Final transition probabilities (p matrix):
[[0.9522 0.0478]
 [1.     0.    ]]

Local crossing probabilities computed successfully
ploc till intf 1:  0.04783439690016397
None ha_factors for 0th ensemble
None
weights:
accepted      60538
rejected      39463
omitted       0
total trajs   100001
total weights 100001
Sum weights ensemble 0:  0.0
None ha_factors for 1th ensemble
None
weights:
accepted      65157
rejected      34844
omitted       0
total trajs   100001
total weights 100001
Sum weights ensemble 1:  99997.0
None ha_factors for 2th ensemble
None
we

In [82]:
print(plocMSM)
for pp in plocMSM:
    print(pp)
print("\n\n")
pcrosslocMSM = np.empty(len(plocMSM))

for i in range (len(pcrosslocMSM)):
    pcrosslocMSM[i] = plocMSM[i]/np.prod(pcrosslocMSM[:i])
    print(pcrosslocMSM[i])
    

[1.         0.82815042 0.81389948 0.79838387 0.65067041 0.63958266]
1.0
0.8281504232252875
0.8138994771000272
0.7983838685343494
0.6506704124653093
0.6395826553421684



1.0
0.8281504232252875
0.9827918386254527
0.9809367016416317
0.8149844180341366
0.9829594877671927


# Error analysis (recursive block analysis)
## Don't run if not needed

In [25]:
from tistools import block_error_analysis_staple
import logging
import sys
from datetime import datetime
from contextlib import redirect_stdout, redirect_stderr

# Create a log file with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_filename = f"block_error_analysis_{timestamp}.log"

# Configure logging to file
file_handler = logging.FileHandler(log_filename, mode='w')
formatter = logging.Formatter('%(message)s')
file_handler.setFormatter(formatter)

# Create logger
log = logging.getLogger()
log.setLevel(logging.INFO)
log.handlers = [file_handler]  # Replace any existing handlers

# Write initial message to log file only
log.info(f"Starting block error analysis at {timestamp}")

# Open the log file for capturing all output
with open(log_filename, 'a') as log_file, redirect_stdout(log_file), redirect_stderr(log_file):
    # Run the block error analysis for StapleTIS
    block_error_analysis_staple(pathensembles, interfaces, 1, pl=False)
    
    # Calculate REPPTIS probabilities for comparison
    repptisploc = []
    for i, pe in enumerate(pathensembles):
        # REPPTIS p_loc
        repptisploc.append(get_local_probs(pe, tr=False))

    _, _, reppfull = get_global_probs_from_dict(repptisploc)
    
    # Write REPPTIS results to log file
    print("\nREPPTIS global crossing probabilities:")
    for i, p in enumerate(reppfull):
        print(f"Interface {i}: {p:.8f}")

# Log file path is the only output to notebook
log.info(f"Analysis complete - results saved to {log_filename}")


NameError: name 'calculate_running_estimate_staple' is not defined

In [ ]:
# PyRETIS code for block error analysis
 
def block_error(data, maxblock=None, blockskip=1):
    """
    Perform block error analysis to estimate the standard deviation in the input data.

    Parameters
    ----------
    data : numpy.array
        The data to analyze.
    maxblock : int, optional
        Maximum block length to consider. Defaults to half the length of the input data.
    blockskip : int, optional
        Skip factor for block lengths. Defaults to 1 (all block lengths considered).

    Returns
    -------
    blocklen : numpy.array
        Array of block lengths considered.
    block_avg : numpy.array
        Block averages as a function of block length.
    block_err : numpy.array
        Standard error estimates as a function of block length.
    block_err_avg : float
        Average error estimate for block lengths greater than maxblock//2.
    """
    n = len(data)
    maxblock = min(maxblock or n // 2, n // 2)
    
    blocklen = np.arange(1, maxblock + 1, blockskip, dtype=np.int_)
    n_blocks = len(blocklen)
    
    block = np.zeros(n_blocks)
    nblock = np.zeros(n_blocks)
    block_avg = np.zeros(n_blocks)
    block_var = np.zeros(n_blocks)

    for i, val in enumerate(data):
        block += val
        full_blocks = (i + 1) % blocklen == 0
        block[full_blocks] /= blocklen[full_blocks]
        nblock[full_blocks] += 1
        deltas = block[full_blocks] - block_avg[full_blocks]
        block_avg[full_blocks] += deltas / nblock[full_blocks]
        block_var[full_blocks] += deltas * (block[full_blocks] - block_avg[full_blocks])
        block[full_blocks] = 0.0

    block_var /= (nblock - 1)
    block_err = np.sqrt(block_var / nblock)
    
    large_blocks = blocklen > maxblock // 2
    block_err_avg = np.mean(block_err[large_blocks])
    
    return blocklen, block_avg, block_err, block_err_avg, maxblock, n//maxblock


def block_error_corr(data, maxblock=None, blockskip=1):
    """
    Run block error analysis and calculate correlation length estimates.

    Parameters
    ----------
    data : numpy.array
        Data to analyze.
    maxblock : int, optional
        Maximum block length to consider. Defaults to half the length of the input data.
    blockskip : int, optional
        Skip factor for block lengths. Defaults to 1 (all block lengths considered).

    Returns
    -------
    blen : numpy.array
        Block lengths considered.
    berr : numpy.array
        Error estimates as a function of block length.
    berr_avg : float
        Average error estimate for blocks with length > maxblock // 2.
    rel_err : numpy.array
        Relative error normalized by the overall average as a function of block length.
    avg_rel_err : float
        Average relative error for blocks with length > maxblock // 2.
    ncor : numpy.array
        Estimated correlation length as a function of block length.
    avg_ncor : float
        Average correlation length for blocks with length > maxblock // 2.
    """
    blen, bavg, berr, berr_avg, max_block_size, min_block_number = block_error(data, maxblock=maxblock, blockskip=blockskip)
    rel_err = berr / abs(bavg[0])
    avg_rel_err = berr_avg / abs(bavg[0])
    ncor = (berr / berr[0])**2
    avg_ncor = (berr_avg / berr[0])**2
    
    return blen, berr, berr_avg, rel_err, avg_rel_err, ncor, avg_ncor, max_block_size, min_block_number

def pathensembles_nskip(obj, nskip):
    keys = [
        'cyclenumbers', 'flags', 'generation', 'lambmaxs', 'lambmins',
        'lengths', 'lmrs', 'newpathnumbers', 'pathnumbers',
        'shootlinks', 'weights', 'dirs', 'istar_idx']
    for key in keys:
        attr = getattr(obj, key)
        setattr(obj, key, attr[:nskip])

In [ ]:
%%capture
from datetime import datetime
import copy

start_cyc = 1
n_cycle = 100000
nstep = 1

ploc_MSM_stored = []
ploc_repptis_stored = []
# This loops over the npy file and calculates tau from cycle 100 every 10 cycles
for nskip in range(start_cyc, n_cycle, nstep):
    pathensemblesi = copy.deepcopy(pathensembles)
    for i, pe in enumerate(pathensemblesi):
        pathensembles_nskip(pe,nskip)
    # Analysis output is saved to the data dictionary.
    data = {}
    for i, pe in enumerate(pathensemblesi):
        if i == 0:
            data[i] = {}
            continue  #  [0-] is not used for Pcross calculations
        
        # Classify the paths according to their path type.
        pathtypes = ("LML", "LMR", "RML", "RMR")
        pathtype_cycles = {}
        for ptype in pathtypes:
            pathtype_cycles[ptype] = unwrap_by_weight(
                    (pe.lmrs == ptype).astype(int), pe.weights)
        
        data[i] = {}
        plocfull = get_local_probs(pe, tr=False)
        data[i]["full"] = {}
        for ptype in pathtypes:
            data[i]["full"][ptype] = plocfull[ptype]

    psfull = []
    for i in range(1, len(pathensemblesi)):   # do not use the 0- ensemble
        psfull.append({"LMR": data[i]["full"]["LMR"], 
                "RML": data[i]["full"]["RML"], 
                "RMR": data[i]["full"]["RMR"],
                "LML": data[i]["full"]["LML"]})

    Pminfull, Pplusfull, Pcrossfull = get_global_probs_from_dict(psfull)

    ploc_repptis_stored.append(Pcrossfull)
    
    N = len(interfaces)
    NS = 2*N

    wi = compute_weight_matrices(pathensemblesi, interfaces, len(interfaces), True)
    pi, _ = get_transition_probzz(wi)
    # pi = get_transition_probs(wi)
    # pi = get_simple_probs(wi)
    M = construct_M_istar(pi, max(4, 2*len(interfaces)), len(interfaces))

    plocMSM = np.ones(len(interfaces))
    for lint in range(1, len(interfaces)):
        # Mi = M[:min(NS, 1+2*lint), :min(NS, 1+2*lint)]
        Mi = M[np.r_[0:2+lint, 1+N:N+lint, -1]]
        Mi = Mi[:, np.r_[0:2+lint, 1+N:N+lint+1]]
        Msum = M[np.r_[0:2+lint, 1+N:N+lint+1]]
        Mi[:, -1] = np.sum(Msum[:, N+lint:], axis=1)
        
        # wi2 = compute_weight_matrices(pathensemblesi[:lint+1], interfaces[:lint+1], len(interfaces), True)
        # pi2 = get_transition_probzz(wi2)
        # # pi = get_transition_probs(wi)
        # # pi = get_simple_probs(wi)
        # Mi2 = construct_M_istar(pi2, max(4, 2*len(interfaces[:lint+1])), len(interfaces[:lint+1]))

        # print(Mi == Mi2)

        z1, z2, y1, y2 = global_pcross_msm_star(Mi)
        plocMSM[lint] = y1[0][0]
    # print(f"plocs: ", plocMSM)

    pcrosslocMSM = np.empty(len(plocMSM))

    for i in range (len(pcrosslocMSM)):
        pcrosslocMSM[i] = plocMSM[i]/np.prod(pcrosslocMSM[:i])
    # print(pcrosslocMSM)

    ploc_MSM_stored.append(plocMSM)

    # for i,fol in enumerate(folders):
    #     set_tau_distrib(pathensemblesi[i])
    #     if True:
    #         set_tau_first_hit_M_distrib(pathensemblesi[i])

    # Compute taus for pathlength analysis
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"{nskip:5d} cycles, Plocs {plocMSM}")

    # Not sure if we need this, need to check later
    del data
    del Mi, wi, pi, z1, z2, y1, y2,N, NS
    del Pcrossfull, Pminfull, Pplusfull
    del plocMSM, pcrosslocMSM

np.save('plocMSM_vs_cycle_interval_1_full.npy', ploc_MSM_stored)
np.save('plocREPPTIS_vs_cycle_interval_1_full.npy', ploc_repptis_stored)

In [ ]:
# Adapted AG, Jan 17, 2025

#----------------
# part Titus
#----------------

# SETTINGS

minblocks = 5 #the minimal number of blocks
filerunav = "plocMSM_vs_cycle_interval_1_full.npy"

# FUNCTIONS

def get_second_column(file):
    arr = []
    with open(file, "r") as f:
        for line in f:
            values = line.strip().split()
            arr.append(float(values[1]))
    return arr
 
def get_first_column(file):
    arr = []
    with open(file, "r") as f:
        for line in f:
            values = line.strip().split()
            arr.append(float(values[0]))
    return arr
 
def rec_blocks(r):
    # no longer used
    """Compute the vector b, which is ... """
    b = np.zeros(len(r))
    for i in range(len(r)):
        if i == 0:
            b[i] = r[i]
        else:
            b[i] = (i+1) * r[i] - i * r[i - 1]
    return b
 
def rec_blocks_from_runav(runav, n):
    # see paper Vervust et al. PyRETIS3, 2024
    # paper: select values k[j*m], indexing starts at 1
    # here: runav, python indexing, so start index n-1 (end of first block)
    # collect them in runav_red
    assert n>0   # block size
    runav_red = runav[n-1::n]
    nb = len(runav_red)  # number of blocks
 
    # array b with average of each block
    b = np.zeros(nb)
    for i in range(nb):
        if i == 0:
            b[i] = runav_red[i]
        else:
            b[i] = (i+1) * runav_red[i] - i * runav_red[i - 1]
    return b
 
def compute_absolute_error(runavfull, bestav, n):
    # compute average in each block
    assert n>0
    blocks = rec_blocks_from_runav(runavfull, n)
    # compute standard deviation between blocks
    sum_qudiff = np.sum((blocks-bestav)**2)
    nb = len(blocks)
    assert nb>1
    Aerr2 = sum_qudiff/(nb*(nb-1))
    Aerr = np.sqrt(Aerr2)  #esitimate of absolute error
    return Aerr
 
 
# GET STARTED
 
# runavfull = get_second_column(filerunav) #all the running average data in an array
runavfull = np.load(filerunav)
runavfull = runavfull[5:,:]
maxbll = int(len(runavfull)/minblocks) #maximum block length
bestav = runavfull[-1] #most accurate average we have
print("maximum block length = ", maxbll)
 
# loop over block size
sizes = np.arange(1,maxbll+1)
if len(np.shape(runavfull)) == 1:
    rel_errors = np.zeros(len(sizes)) if len(np.shape(runavfull)) == 1 else np.zeros((len(sizes), np.shape(runavfull)[1]))
    for i, n in enumerate(sizes):
        # n is block size
        Aerr = compute_absolute_error(runavfull, bestav, n)
        Rerr = Aerr/bestav
        rel_errors[i] = Rerr
    
    second_half = rel_errors[len(rel_errors)//2:]
    half_av_err = np.mean(second_half)
    print("estimated relative error=", half_av_err)
    Nstatineff=(half_av_err/rel_errors[0])**2
    print("statistical inefficiency=", Nstatineff)
    
    plt.figure()
    plt.plot(sizes, rel_errors)
    plt.axhline(y=half_av_err, color='r', linestyle='--')
    plt.xlabel('Block Length')
    plt.ylabel('Relative Error')
    plt.title('Error Analysis')
    plt.show()
    # plt.savefig("figure_titus.png")

elif len(np.shape(runavfull)) == 2:
    rel_errs = np.empty(len(interfaces))
    for loc in range(len(interfaces)):
        rel_errors = np.zeros(len(sizes))
        intf_i = runavfull[:,loc]
        bestav = intf_i[-1]
        for i, n in enumerate(sizes):
        # n is block size
            Aerr = compute_absolute_error(intf_i, bestav, n)
            Rerr = Aerr/bestav
            rel_errors[i] = Rerr

        second_half = rel_errors[len(rel_errors)//2:]
        half_av_err = np.mean(second_half)
        rel_errs[loc] = half_av_err
        print("estimated relative error=", half_av_err)
        Nstatineff=(half_av_err/rel_errors[0])**2
        print("statistical inefficiency=", Nstatineff)
        
        plt.figure()
        plt.plot(sizes, rel_errors)
        plt.axhline(y=half_av_err, color='r', linestyle='--')
        plt.xlabel('Block Length')
        plt.ylabel('Relative Error')
        if loc < len(interfaces)-1:
            plt.title('Error Analysis $p_{local}$'+f' - interface {loc}')
        else:
            plt.title('Error Analysis $P_{cross}$')
        plt.show()
        # plt.savefig("figure_titus.png")

In [ ]:
stored_values = np.load('plocREPPTIS_vs_cycle_interval_10.npy')

if len(np.shape(stored_values)) == 1:
    stored_values = stored_values[~np.isnan(stored_values)] # remove nans in the beginning
    blen, berr, berr_avg, rel_err, avg_rel_err, ncor, avg_ncor, max_block_size, min_block_number = block_error_corr(stored_values,10)
    print("=" * 60)
    print(indir[57:-9])
    print(f"Total Data Points: {len(stored_values)}, max block size: {max_block_size}, min block number: {min_block_number}")
    print(f"Average Relative Error for blocks > maxblock/2: {avg_rel_err * 100:.1f}%")
    print(f"Average Correlation Length for Large Blocks: {int(avg_ncor)}")
    plt.figure(figsize=(8, 6))
    plt.plot(rel_err, marker='o', linestyle='-', label = indir[57:-9])
    plt.xlabel("Block Size")
    plt.ylabel("Rlative Error")
    plt.title(f"Ave Rel Err (blocks > maxblock/2): {avg_rel_err * 100:.1f}%, Block interval 10 cycles")
    plt.grid(True)
    plt.legend()
    # plt.savefig("Block_Error_Tau.png", dpi=1000, bbox_inches='tight')
elif len(np.shape(stored_values)) == 2:
    rel_errs = np.empty(len(interfaces))
    for loc in range(1,len(interfaces)):
        intf_i = stored_values[:,loc]
        blen, berr, berr_avg, rel_err, avg_rel_err, ncor, avg_ncor, max_block_size, min_block_number = block_error_corr(intf_i[~np.isnan(intf_i)],1000)
        rel_errs[loc] = avg_rel_err

        print("=" * 60)
        print(indir[57:-9])
        print(f"Total Data Points: {len(stored_values)}, max block size: {max_block_size}, min block number: {min_block_number}")
        print(f"Average Relative Error for blocks > maxblock/2: {avg_rel_err * 100:.10f}%")
        print(f"Average Correlation Length for Large Blocks: {int(avg_ncor)}")
        plt.figure(figsize=(8, 6))
        plt.plot(rel_err, linestyle='-', label = indir[57:-9])
        plt.xlabel("Block Size")
        plt.ylabel("Rlative Error")
        plt.title(f"Ave Rel Err (blocks > maxblock/2): {avg_rel_err * 100:.1f}%, Block interval 10 cycles")
        plt.grid(True)
        plt.legend()
        # plt.savefig("Block_Error_Tau.png", dpi=1000, bbox_inches='tight')

# Collecting times

In [25]:
import numpy as np
from tistools import (
    set_taus_staple, 
    mfpt_istar_balanced,
    mfpt_istar,
    construct_tau_matrix_staple,
    mfpt_to_absorbing_staple_balanced,
    mfpt_to_absorbing_staple
)

print("=" * 80)
print("COMPUTING TAU MATRICES FOR PATH ENSEMBLES")
print("=" * 80)

# 1. Compute and collect tau, tau1, tau2, and taum matrices
# This replaces `compute_path_taus` and manually aggregating them
tau_data = set_taus_staple(pathensembles, interfaces)

print("\nTau matrices computed:")
for key, mat in tau_data.items():
    if isinstance(mat, np.ndarray):
        print(f"  {key:<5} - shape: {mat.shape}, non-zero elements: {np.count_nonzero(mat)}")

print("\n" + "=" * 80)
print("COMPUTING MEAN FIRST PASSAGE TIME (MFPT) AND FLUX")
print("=" * 80)

# 2. Calculate MFPT and Flux using both balanced and non-balanced iSTAR approaches
# M_valid, N, and NS are already constructed above

# Fallbacks for time conversion factors if not globally defined
dt_val = globals().get('dt', 0.005)
subc_val = globals().get('subc', 1.0)
xi_val = globals().get('xi_val', None)

# time scale factor: assumes reference uses dt * subc * 1e-12 or just dt
time_factor = dt_val * subc_val

pcross_global = None
if 'y1' in globals():
    pcross_global = y1[0][0]

try:
    # Base MFPT returning to interface 0
    g1, g2, h1, h2 = mfpt_istar_balanced(M_valid, tau_data, doprint=False)
    g1_nb, g2_nb, h1_nb, h2_nb = mfpt_istar(M_valid, tau_data, doprint=False)
    
    mfpt_0 = h1[0][0]
    mfpt_nb_0 = h1_nb[0][0]

    # Prepare matrices for absorbing state (state B)
    absor = np.array([NS - 1])
    kept = np.array([i for i in range(NS) if i not in absor])

    if "taum" not in tau_data:
        taumm = tau_data['tau'] - tau_data['tau1'] - tau_data['tau2']
    else:
        taumm = tau_data['taum'].copy()
    # Handle possible xi_val factor the same way as the reference
    taumm[0,0] /= xi_val if xi_val is not None else taumm[0,0]
    
    tau1m = construct_tau_matrix_staple(tau_data['tau1'], N)
    taummm = construct_tau_matrix_staple(taumm, N)
    tau2m = construct_tau_matrix_staple(tau_data['tau2'], N)

    # Compute AB passages
    _, _, h1mfpt, _ = mfpt_to_absorbing_staple_balanced(M_valid, tau_data['tau1'], taumm, tau_data['tau2'], absor, kept)
    _, _, h1mfpt_nb, _ = mfpt_to_absorbing_staple(M_valid, tau1m, taummm, tau2m, absor, kept)
    
    mfpt_AB = h1mfpt[0][0] * time_factor
    mfpt_nb_AB = h1mfpt_nb[0][0] * time_factor

    # Calculate flux (from state A)
    if xi_val is not None:
        flux = 1 / ((tau_data['tau'][0,0]/xi_val + mfpt_0) * time_factor)
        flux_nb = 1 / ((tau_data['tau'][0,0]/xi_val + mfpt_nb_0) * time_factor)
    else:
        flux = 1 / ((tau_data['tau'][0,0] + mfpt_0) * time_factor)
        flux_nb = 1 / ((tau_data['tau'][0,0] + mfpt_nb_0) * time_factor)

    # Output Comparisons
    print(f"Time factor used (dt * subc): {time_factor}")
    if xi_val is not None:
        print(f"xi factor applied: {xi_val:.4g}")
    else:
        print(f"xi factor applied: None")

    print("\n" + "-" * 40)
    print(f" Tau[0-]:     {tau_data['tau'][0,0]:.4e}")
    
    print("\nTau[0+] Comparison:")
    print(f"  Balanced Tau[0+]:     {mfpt_0:.4e}")
    print(f"  Non-balanced Tau[0+]: {mfpt_nb_0:.4e}")

    print("MFPT Comparison (A -> B):")
    print(f"  Balanced MFPT:     {mfpt_AB:.4e}")
    print(f"  Non-balanced MFPT: {mfpt_nb_AB:.4e}")

    print("\nFlux Comparison:")
    print(f"  Balanced Flux:     {flux:.6e}")
    print(f"  Non-balanced Flux: {flux_nb:.6e}")
    print("-" * 40)

    if pcross_global is not None:
        print("\nRate Comparison (1/MFPT vs Flux * Pcross):")
        print(f"  Balanced rate:     {1/mfpt_AB:.6e}  vs  {flux * pcross_global:.6e}")
        print(f"  Non-balanced rate: {1/mfpt_nb_AB:.6e}  vs  {flux_nb * pcross_global:.6e}")
except Exception as e:
    print(f"Error calculating MFPT/Flux: {e}")


COMPUTING TAU MATRICES FOR PATH ENSEMBLES

Tau matrices computed:
  tau1  - shape: (6, 5), non-zero elements: 12
  tau2  - shape: (6, 5), non-zero elements: 11
  tau   - shape: (6, 5), non-zero elements: 21

COMPUTING MEAN FIRST PASSAGE TIME (MFPT) AND FLUX
Time factor used (dt * subc): 0.005
xi factor applied: None

----------------------------------------
 Tau[0-]:     1.1325e+02

Tau[0+] Comparison:
  Balanced Tau[0+]:     2.9931e+02
  Non-balanced Tau[0+]: 2.5939e+02
MFPT Comparison (A -> B):
  Balanced MFPT:     2.0450e+00
  Non-balanced MFPT: 2.0605e+00

Flux Comparison:
  Balanced Flux:     4.847809e-01
  Non-balanced Flux: 5.367076e-01
----------------------------------------

Rate Comparison (1/MFPT vs Flux * Pcross):
  Balanced rate:     4.889918e-01  vs  3.055654e-01
  Non-balanced rate: 4.853235e-01  vs  3.382957e-01
